In [2]:
!pip install opencv-python mediapipe

In [4]:
import cv2
import mediapipe as mp
import math

# Initialize MediaPipe
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(max_num_faces=1)
mp_drawing = mp.solutions.drawing_utils

# Angle calculation helper
def calculate_angle(a, b, c):
    ab = [a[0] - b[0], a[1] - b[1]]
    cb = [c[0] - b[0], c[1] - b[1]]
    dot = ab[0]*cb[0] + ab[1]*cb[1]
    ab_len = math.hypot(ab[0], ab[1])
    cb_len = math.hypot(cb[0], cb[1])
    angle = math.acos(dot / (ab_len * cb_len + 1e-6))
    return math.degrees(angle)

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Process face and pose
    face_result = face_mesh.process(rgb)
    pose_result = pose.process(rgb)

    posture_msg = "Detecting..."
    hand_msg = "Checking hand posture..."
    color = (255, 255, 255)

    #### FACE & NECK POSTURE ####
    if face_result.multi_face_landmarks:
        face_landmarks = face_result.multi_face_landmarks[0].landmark
        try:
            forehead = (int(face_landmarks[10].x * w), int(face_landmarks[10].y * h))
            nose = (int(face_landmarks[1].x * w), int(face_landmarks[1].y * h))
            chin = (int(face_landmarks[152].x * w), int(face_landmarks[152].y * h))

            # Draw points
            cv2.circle(frame, forehead, 4, (0, 255, 0), -1)
            cv2.circle(frame, nose, 4, (255, 0, 0), -1)
            cv2.circle(frame, chin, 4, (0, 255, 0), -1)

            face_angle = calculate_angle(forehead, nose, chin)

            if 165 <= face_angle <= 195:
                posture_msg = "Face & Neck: Straight ✅"
                color = (0, 255, 0)
            else:
                posture_msg = "Face/Neck: Incorrect ❌"
                color = (0, 0, 255)

            cv2.putText(frame, f"Face Angle: {int(face_angle)}°", (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            cv2.putText(frame, posture_msg, (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

        except IndexError:
            pass

    #### HAND/ARM POSTURE ####
    if pose_result.pose_landmarks:
        lms = pose_result.pose_landmarks.landmark
        try:
            # Get points
            l_shoulder = (int(lms[mp_pose.PoseLandmark.LEFT_SHOULDER].x * w), int(lms[mp_pose.PoseLandmark.LEFT_SHOULDER].y * h))
            l_elbow = (int(lms[mp_pose.PoseLandmark.LEFT_ELBOW].x * w), int(lms[mp_pose.PoseLandmark.LEFT_ELBOW].y * h))
            l_wrist = (int(lms[mp_pose.PoseLandmark.LEFT_WRIST].x * w), int(lms[mp_pose.PoseLandmark.LEFT_WRIST].y * h))

            r_shoulder = (int(lms[mp_pose.PoseLandmark.RIGHT_SHOULDER].x * w), int(lms[mp_pose.PoseLandmark.RIGHT_SHOULDER].y * h))
            r_elbow = (int(lms[mp_pose.PoseLandmark.RIGHT_ELBOW].x * w), int(lms[mp_pose.PoseLandmark.RIGHT_ELBOW].y * h))
            r_wrist = (int(lms[mp_pose.PoseLandmark.RIGHT_WRIST].x * w), int(lms[mp_pose.PoseLandmark.RIGHT_WRIST].y * h))

            # Arm angles
            left_angle = calculate_angle(l_shoulder, l_elbow, l_wrist)
            right_angle = calculate_angle(r_shoulder, r_elbow, r_wrist)

            if 160 <= left_angle <= 180 and 160 <= right_angle <= 180:
                hand_msg = "Hand Posture: Correct ✅"
                color = (0, 255, 0)
            else:
                hand_msg = "Hand Posture: Incorrect ❌"
                color = (0, 0, 255)

            # Draw text
            cv2.putText(frame, f"L-Arm: {int(left_angle)}°, R-Arm: {int(right_angle)}°", (20, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            cv2.putText(frame, hand_msg, (20, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

            # Draw pose skeleton
            mp_drawing.draw_landmarks(frame, pose_result.pose_landmarks, mp_pose.POSE_CONNECTIONS)

        except IndexError:
            pass

    # Show frame
    cv2.imshow("Face + Hand Posture Detection (Press 'q' to Quit)", frame)

    # Exit on 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Exiting webcam...")
        break

cap.release()
cv2.destroyAllWindows()

Exiting webcam...
